# Quantum-Inspired Optimization for Logistics Routing
## TSP Research Analysis: Brute-Force vs Genetic Algorithm

**Goal:** verify the TSP workflow, characterize brute-force scaling, tune the Genetic Algorithm, and motivate QAOA.

**Contents**
1. Problem instance visualisation + distance matrix
2. Brute-force solution landscape (all 24 routes)
3. GA convergence behaviour
4. GA hyperparameter sensitivity
5. Scaling study: runtime + solution quality vs N
6. Statistical comparison (30 trials)
7. Route gallery across seeds
8. Conclusions + QAOA roadmap

In [ ]:
import os, sys, time, math, random
from itertools import permutations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Make project modules importable from notebooks/ dir
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from core.city_generator import generate_cities
from core.distance import calculate_distance
from algorithms.tsp_bruteforce import solve_tsp_bruteforce
from algorithms.genetic_algorithm import solve_tsp_genetic
from utils.metrics import optimality_gap, distance_matrix, route_leg_distances

print('imports OK, root =', ROOT)

## 1. Problem instance: 5 random cities

Each city is a point in [0, 100]^2. The route must start at City 1, visit every city once, return to City 1.

In [ ]:
cities = generate_cities(n=5, seed=42)
for i, (x, y) in enumerate(cities):
    print(f'City {i+1}: ({x:.2f}, {y:.2f})')

D = distance_matrix(cities)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: scatter map
ax = axes[0]
ax.scatter(cities[:, 0], cities[:, 1], c='red', s=100, zorder=5)
for i, (x, y) in enumerate(cities):
    ax.annotate(f'City {i+1}', (x, y), textcoords='offset points', xytext=(6, 6), fontsize=10)
ax.scatter([cities[0, 0]], [cities[0, 1]], c='green', s=160, zorder=6, label='Depot (City 1)')
ax.set_title('5-city TSP instance (seed=42)')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.legend(); ax.grid(alpha=0.3)

# Right: distance matrix heatmap
sns.heatmap(D, annot=True, fmt='.1f', cmap='viridis', ax=axes[1],
            xticklabels=[f'C{i+1}' for i in range(5)],
            yticklabels=[f'C{i+1}' for i in range(5)])
axes[1].set_title('Pairwise Euclidean distance matrix')
plt.tight_layout(); plt.show()

## 2. Brute-force landscape: all 24 routes

With City 1 fixed as start, there are (5-1)! = 24 permutations. We enumerate all of them to see the full solution landscape.

In [ ]:
all_routes, all_dists = [], []
for perm in permutations(range(1, 5)):
    route = [0] + list(perm)
    all_routes.append(route)
    all_dists.append(calculate_distance(cities, route))

all_dists = np.array(all_dists)
order = np.argsort(all_dists)
print(f'Best : {[r+1 for r in all_routes[order[0]]]} -> {all_dists[order[0]]:.2f}')
print(f'Median: {[r+1 for r in all_routes[order[12]]]} -> {all_dists[order[12]]:.2f}')
print(f'Worst: {[r+1 for r in all_routes[order[-1]]]} -> {all_dists[order[-1]]:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(all_dists, bins=12, color='skyblue', edgecolor='black')
axes[0].axvline(all_dists.min(), color='green', linestyle='--', label=f'optimum {all_dists.min():.1f}')
axes[0].axvline(all_dists.mean(), color='red', linestyle='--', label=f'mean {all_dists.mean():.1f}')
axes[0].set_title('Distribution of all 24 route lengths')
axes[0].set_xlabel('Total distance'); axes[0].set_ylabel('Count'); axes[0].legend()

axes[1].plot(np.sort(all_dists), marker='o', markersize=4)
axes[1].set_title('Sorted route lengths (solution landscape)')
axes[1].set_xlabel('Route rank (0=best)'); axes[1].set_ylabel('Distance'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Best vs worst route (same cities, different order)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, idx, title in [(axes[0], order[0], 'BEST'), (axes[1], order[-1], 'WORST')]:
    route = all_routes[idx]
    rc = cities[route + [route[0]]]
    ax.plot(rc[:, 0], rc[:, 1], 'b-', linewidth=2, alpha=0.7)
    ax.scatter(cities[:, 0], cities[:, 1], c='red', s=90, zorder=5)
    for i, (x, y) in enumerate(cities):
        ax.annotate(f'C{i+1}', (x, y), textcoords='offset points', xytext=(5, 5))
    ax.scatter([cities[0,0]], [cities[0,1]], c='green', s=150, zorder=6)
    ax.set_title(f"{title}: {'->'.join(str(r+1) for r in route)} = {all_dists[idx]:.1f}")
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. GA convergence: does it reliably find the optimum?

Run GA 5 times on the same instance and overlay the best-distance-per-generation curves.

In [ ]:
bf_route, bf_dist = solve_tsp_bruteforce(cities)
print(f'Optimum (brute-force): {bf_dist:.2f}')

histories = []
for run in range(5):
    random.seed(run); np.random.seed(run)
    _, d, h = solve_tsp_genetic(cities, population_size=50, generations=100, mutation_rate=0.05)
    histories.append(h)
    print(f'run {run}: final={d:.2f} gap={optimality_gap(d, bf_dist):.2f}%')

plt.figure(figsize=(9, 4.5))
for i, h in enumerate(histories):
    plt.plot(h, alpha=0.6, label=f'run {i}')
plt.axhline(bf_dist, color='black', linestyle='--', label=f'optimum {bf_dist:.1f}')
plt.title('GA convergence: 5 independent runs (pop=50, gen=100)')
plt.xlabel('Generation'); plt.ylabel('Best distance so far')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 4. Hyperparameter sensitivity

Vary population size and mutation rate; measure final optimality gap over repeated trials.

In [ ]:
test_cities = generate_cities(n=7, seed=7)
_, opt7 = solve_tsp_bruteforce(test_cities)
print(f'7-city optimum: {opt7:.2f}')

records = []
for pop in [20, 50, 100]:
    for mut in [0.01, 0.05, 0.15]:
        for rep in range(5):
            random.seed(rep*17); np.random.seed(rep*17)
            _, d, _ = solve_tsp_genetic(test_cities, population_size=pop, generations=60, mutation_rate=mut)
            records.append({'pop': pop, 'mut': mut, 'gap': optimality_gap(d, opt7)})
df_hp = pd.DataFrame(records)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=df_hp, x='pop', y='gap', ax=axes[0], palette='Blues')
axes[0].set_title('Gap vs population size (n=7)'); axes[0].set_xlabel('Population'); axes[0].set_ylabel('Gap %')
sns.boxplot(data=df_hp, x='mut', y='gap', ax=axes[1], palette='Oranges')
axes[1].set_title('Gap vs mutation rate (n=7)'); axes[1].set_xlabel('Mutation rate'); axes[1].set_ylabel('Gap %')
plt.tight_layout(); plt.show()
print(df_hp.groupby(['pop','mut'])['gap'].mean().round(2).to_string())

## 5. Scaling study: runtime + quality vs number of cities

Uses `results/benchmark_results.csv` generated by `python -m experiments.benchmark`. Falls back to a quick live run if missing.

In [ ]:
csv_path = os.path.join(ROOT, 'results', 'benchmark_results.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f'loaded {len(df)} records from results/benchmark_results.csv')
else:
    print('CSV not found — running quick live benchmark (4..8 cities, 5 trials)')
    rows = []
    for n in range(4, 9):
        for t in range(5):
            c = generate_cities(n=n, seed=1000+t*100+n)
            s = time.perf_counter(); _, b = solve_tsp_bruteforce(c); tb = (time.perf_counter()-s)*1000
            s = time.perf_counter(); _, g, _ = solve_tsp_genetic(c, 30, 50, 0.05); tg = (time.perf_counter()-s)*1000
            rows += [{'algorithm':'brute_force','n_cities':n,'distance':b,'time_ms':tb,'gap':0.0},
                       {'algorithm':'ga_default','n_cities':n,'distance':g,'time_ms':tg,'gap':optimality_gap(g,b)}]
    df = pd.DataFrame(rows)

g = df.groupby(['algorithm','n_cities']).agg(mean_time=('time_ms','mean'), mean_gap=('gap','mean')).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for algo, sub in g.groupby('algorithm'):
    axes[0].plot(sub['n_cities'], sub['mean_time'], marker='o', label=algo)
axes[0].set_yscale('log')
axes[0].set_title('Mean runtime vs N (log scale)')
axes[0].set_xlabel('Cities (N)'); axes[0].set_ylabel('Time (ms, log)'); axes[0].legend()

ga_only = g[g['algorithm'].str.startswith('ga')]
sns.barplot(data=ga_only, x='n_cities', y='mean_gap', hue='algorithm', ax=axes[1])
axes[1].set_title('GA mean optimality gap vs N')
axes[1].set_xlabel('Cities (N)'); axes[1].set_ylabel('Gap %')
plt.tight_layout(); plt.show()
print(g.round(2).to_string())

## 6. Statistical comparison: 30 trials at N=7

Is GA statistically close to optimal? Boxplot of distances + success rate within 1%.

In [ ]:
N_TRIALS, N = 30, 7
rows = []
for t in range(N_TRIALS):
    c = generate_cities(n=N, seed=5000+t)
    _, b = solve_tsp_bruteforce(c)
    _, g, _ = solve_tsp_genetic(c, population_size=50, generations=100, mutation_rate=0.05)
    rows.append({'trial': t, 'brute_force': b, 'genetic': g, 'gap': optimality_gap(g, b)})
df30 = pd.DataFrame(rows)

sr = float((df30['gap'] <= 1.0).mean())
print(f"mean gap: {df30['gap'].mean():.2f}% | success within 1%: {sr*100:.0f}% | worst gap: {df30['gap'].max():.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=df30[['brute_force','genetic']], ax=axes[0], palette='Set2')
axes[0].set_title(f'Distance distribution over {N_TRIALS} trials (N={N})')
axes[0].set_ylabel('Distance')
axes[1].hist(df30['gap'], bins=12, color='salmon', edgecolor='black')
axes[1].axvline(1.0, color='green', linestyle='--', label='1% tolerance')
axes[1].set_title('GA optimality gap distribution')
axes[1].set_xlabel('Gap %'); axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Route gallery: same algorithm, different maps

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, seed in zip(axes.flat, [1, 2, 3, 4]):
    c = generate_cities(n=6, seed=seed)
    r, d = solve_tsp_bruteforce(c)
    rc = c[r + [r[0]]]
    ax.plot(rc[:, 0], rc[:, 1], 'b-', linewidth=2, alpha=0.7)
    ax.scatter(c[:, 0], c[:, 1], c='red', s=80, zorder=5)
    for i, (x, y) in enumerate(c):
        ax.annotate(f'C{i+1}', (x, y), textcoords='offset points', xytext=(4, 4), fontsize=9)
    ax.scatter([c[0,0]], [c[0,1]], c='green', s=130, zorder=6)
    ax.set_title(f"seed={seed}: {'->'.join(str(x+1) for x in r)} = {d:.1f}")
    ax.grid(alpha=0.3)
plt.suptitle('Optimal 6-city routes across 4 random maps', fontsize=13)
plt.tight_layout(); plt.show()

## 8. Conclusions + QAOA roadmap

### Findings
- Brute-force is exact but factorial: (N-1)! routes. Fine at N=5 (24 routes), painful beyond N=10.
- GA finds the optimum reliably at N<=6 and stays within a few % at N=7..9 with tiny runtime.
- Population 50 + mutation 0.05 is a solid default; larger populations help more than higher mutation.

### Why QAOA next?
- TSP maps naturally to QUBO / Ising Hamiltonians — the native language of QAOA.
- QAOA needs N^2 qubits for N cities (one-hot encoding), so N=5 needs 25 qubits: simulable, real hardware is near-term.
- Research question: can shallow-depth QAOA (p=1..3) match GA solution quality with fewer cost evaluations?

In [ ]:
N = np.arange(4, 13)
bf_routes = [math.factorial(n-1) for n in N]
qubits = N**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(N, bf_routes, marker='o', color='red')
axes[0].set_yscale('log')
axes[0].set_title('Brute-force search space (N-1)!')
axes[0].set_xlabel('Cities N'); axes[0].set_ylabel('Routes (log scale)')
for x, y in zip(N, bf_routes):
    axes[0].annotate(f'{y:,}', (x, y), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=8)
axes[1].bar(N.astype(str), qubits, color='purple', alpha=0.7)
axes[1].set_title('QAOA qubits needed (N^2 one-hot encoding)')
axes[1].set_xlabel('Cities N'); axes[1].set_ylabel('Qubits')
plt.tight_layout(); plt.show()